In [1]:
%pip install nltk 
%pip install ir_measures
%pip install terrier
%pip install polars
%pip install wordninja
%pip install python-terrier

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [2]:

import requests
import tarfile
import os
import joblib
import gc
import re
import string
import nltk
from functools import lru_cache
import unicodedata
import wordninja
from collections import defaultdict, Counter
from tqdm import tqdm
import heapq
import math
import matplotlib.pyplot as plt
# from scipy.stats import ttest_ind
import pandas as pd
import polars as pl
import pickle
import pyterrier as pt




**Purpose**: Installs external libraries required for the project.
1. `nltk`: A library for working with natural language processing (NLP), including tokenization and stemming.
2. `ir_measures`: Used for calculating information retrieval performance metrics like `P@5`, `nDCG`, etc.
3. `python-terrier`: Terrier toolkit for Information Retrieval tasks (BM25 modeling, indexing, etc.).
4. `polars`: A data processing library similar to pandas but optimized for speed and memory usage.
5. `wordninja`: Library for splitting concatenated words based on statistical algorithms.

In [3]:

url = 'https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz'
local_filename = 'collection.tar.gz'

if not os.path.exists(local_filename):
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(local_filename, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

extract_path = os.getcwd()

if not os.path.exists(extract_path):
    with tarfile.open(local_filename, 'r:gz') as tar:
        tar.extractall(path=extract_path)


**Purpose**: Downloads and extracts the MS MARCO ranking dataset (a standard dataset for evaluating search engines). 
- This dataset contains `collection.tar.gz`, which holds the MS MARCO passage and document collections.

In [ ]:

gc.enable()

**Purpose**: Enables Python's garbage collection (GC) to automate memory management.
- Ensures that unreferenced memory is cleared, especially relevant when working with  MS MARCO and other large datasets.

In [5]:

from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

STEMMER, STOPWORDS = SnowballStemmer("english"), set(stopwords.words("english"))

# Gestione di acronimi (U.S.A.) -> USA
# (Mr.Jhonsons) -> Mr.Jhonson
# se manca il punto alla fine, come www.colab.com, non conta, verrà gestito dalla WWW_REGEX
ACRONYM_RE = re.compile(r"\b(?:[A-Za-z]\.){2,}")

HTTP_REGEX = re.compile(r"\s?http\S+")
WWW_REGEX = re.compile(r"\s?www\S+")
EMAIL_REGEX = re.compile(r"\S+@\S+")

CLEAN_TABLE = str.maketrans(string.punctuation, " " * len(string.punctuation))

**Purpose**: Sets up regular expressions and NLP tools for text preprocessing.
1. Initializes an English stemmer (`SnowballStemmer`) to reduce words to their root form.
2. Loads a set of English stopwords from NLTK (e.g., "is", "the", "and").
3. Initializes Regex patterns for preprocessing:
   - `ACRONYM_RE`: Matches acronyms (e.g., "U.S.A.").
   - `HTTP_REGEX`: Removes `http` links.
   - `WWW_REGEX`: Removes URLs starting with `www`.
   - `EMAIL_REGEX`: Detects email addresses.
4. `CLEAN_TABLE`: A mapping to replace punctuation (like `.,%`) with a space.

#### **Theoretical Foundation**:
1. **Preprocessing for Retrieval Systems**:
   - Text preprocessing is essential to:
     - Standardize information.
     - Remove irrelevant content (e.g., stopwords, URLs).
     - Reduce computational overhead in later indexing and retrieval stages by simplifying data.
   - These transformations ensure that both **queries** and **documents** have a consistent format for matching.

2. **Linguistic Tools**:
   - **Stemming with SnowballStemmer**:
     - Reduces words to their root forms. For example:
       - "playing" → "play", "ran" → "run".
     - Stemming ensures that different word forms of the same concept are treated equally during retrieval.
   - **Stopword Removal**:
     - Eliminates common terms (e.g., "and", "is") that carry little meaning in context. These are stored in the `STOPWORDS` set.

3. **Regex Patterns**:
   - Regex is used to handle specific text cleaning tasks:
     - `ACRONYM_RE`: Resolves abbreviations and acronyms by removing dots.
     - `HTTP_REGEX`, `WWW_REGEX`, `EMAIL_REGEX`: Detect and remove URLs and email addresses to prevent irrelevant noise in document queries (frequently found in datasets like MS MARCO).

4. **Punctuation Mapping**:
   - **`str.maketrans`** efficiently replaces all punctuation symbols with spaces. This enhances tokenization by ensuring separators do not fragment words inconsistently.

5. **Relevance to IR Systems**:
   - Preprocessing ensures that indexed documents are in a query-compatible format, and noise removal enhances retrieval precision.


In [6]:


@lru_cache(maxsize=None) # Per l'ottimizzazione del preprocessing
def stem_word(word):
    return STEMMER.stem(word)

@lru_cache(maxsize=None)
def split_word(word):
    return wordninja.split(word)

def preprocess(text):

    text = unicodedata.normalize("NFKD", text).encode('ASCII', 'ignore').decode("utf-8").lower()



    text = ACRONYM_RE.sub(lambda x: x.group().replace('.', ''), text)

    text = HTTP_REGEX.sub(' ', text)
    text = WWW_REGEX.sub(' ', text)
    text = EMAIL_REGEX.sub(' ', text)

    # Rimozione punteggiatura e altri caratteri non utili
    # Viene messa alla fine per dare l'opportunità alle regex di usare i caratteri di punt come @ o i punti per gli acronimi.
    text = text.translate(CLEAN_TABLE)

    words = text.strip().split()
    
    words_with_no_close_duplicate = []
    current = ""
    for word in words:
        if word != current:
            splitted_words = split_word(word) if len(word) > 10 else [word]
            for splitted_word in splitted_words:
                if splitted_word not in STOPWORDS:
                    words_with_no_close_duplicate.append(stem_word(splitted_word))
            current = word
            
    return words_with_no_close_duplicate

**Purpose**: Implements text preprocessing for NLP tasks.
1. **Functions**:
   - `stem_word`: Caches and stems words efficiently for faster reuse.
   - `split_word`: Splits attached or concatenated words (e.g., `joinedsentence` → `joined sentence`).
2. **Preprocessing Steps**:
   - Converts text to lowercase and normalizes Unicode characters (e.g., `Cafè` → `Cafe`).
   - Removes acronyms, URLs, email addresses, and punctuation.
   - Tokenizes and removes duplicate neighboring words.
   - Stems each word unless it's a stopword.

In [7]:

tqdm.pandas(desc='Preprocessing...')

def build_index(dataset):
    lexicon = defaultdict(lambda: [0, 0, 0])  # [termid, document frequency, term frequency]
    doc_index = []
    direct_index = defaultdict(dict)
    inv_d, inv_f = defaultdict(list), defaultdict(list)

    # Altro
    termid = 0
    dataset_size = dataset.shape[0]
    total_toks = 0

    for row in tqdm(dataset.itertuples(index=False), desc='Indexing', total=dataset_size):
        docid = row.docno
        tokens = row.tokens

        token_tf = Counter(tokens)

        for token, tf in token_tf.items():
            if token not in lexicon:
                lexicon[token][0] = termid
                termid += 1

            token_id = lexicon.get(token)[0]
            inv_d[token_id].append(int(docid))
            inv_f[token_id].append(tf)
            direct_index[int(docid)][token_id] = tf
            lexicon[token][1] += 1
            lexicon[token][2] += tf

        doclen = len(tokens)
        doc_index.append((docid, doclen))
        total_toks += doclen

    stats = {
        'num_docs': dataset_size,
        'num_terms': len(lexicon),
        'num_tokens': total_toks,
    }

    return dict(lexicon), {'docids': dict(inv_d), 'freqs': dict(inv_f)}, doc_index, dict(direct_index), stats

**Purpose**: Constructs the index structure for a document retrieval system.
1. **Data Structures**:
   - `lexicon`: Maps tokens to terms, term frequencies, and document frequencies.
   - `direct_index`: Tracks token frequencies for each document.
   - `inv_d` and `inv_f`: Store inverted index data (term to document list and term frequencies).
2. **Statistics**:
   - Computes stats like total tokens, distinct terms, and documents.
3. Iterates through the dataset row-by-row to process all tokens and build efficient retrieval structures.

In [8]:

class InvertedIndex:

    class PostingListIterator:
        def __init__(self, docids, freqs, doclens, N, avgdl, k, b):
            self.docids = docids # [1, 2, 20, etc...]
            self.freqs = freqs # [15, 30, 4, etc...]
            self.doclens = doclens # {0: 30; 1: 20 etc...}
            self.N = N
            self.avgdl = avgdl
            self.K = k
            self.B = b

            self.pos = 0 # Rappresenta la posizione dell'iteratore all'interno della posting list
            """
                                |
                               -V-
            self.docids = [1,   2,   20...]
            """
        def docid(self):
            if self.is_end_list():
                return math.inf
            return self.docids[self.pos]

        def score(self):
            if self.is_end_list():
                return math.inf

            tf_i = self.freqs[self.pos]
            dl_i = self.doclens.get(self.docid())
            df_i = self.len()
                                                    # ---------------- BM25 ----------------- #
            return 0 if df_i == 0 else (tf_i/(self.K * ((1 - self.B) + self.B * dl_i/self.avgdl) + tf_i)) * math.log(self.N / df_i)

        def next(self, target = None):
            if not target:
                if not self.is_end_list():
                    self.pos += 1
            else:
                if target > self.docid():
                    try:
                        self.pos = self.docids.index(target, self.pos)
                    except ValueError:
                        self.pos = len(self.docids)

        def is_end_list(self):
            return self.pos == len(self.docids)

        def len(self):
            return len(self.docids)

    def __init__(self, lex, inv, doc, direct, stats, k, b):
        self.lexicon = lex
        self.inv = inv
        self.doc = doc
        self.direct = direct
        self.stats = stats

        self.N = self.num_docs()
        self.doclens = {int(doc_entry[0]): doc_entry[1] for doc_entry in self.doc}
        total_dl = sum(doc_entry[1] for doc_entry in self.doc)
        self.avgdl = total_dl / self.N

        self.K = k
        self.B = b

    def num_docs(self):
        return self.stats['num_docs']

    def get_posting(self, termid):
        docids = self.inv['docids'].get(termid)
        freqs = self.inv['freqs'].get(termid)

        return InvertedIndex.PostingListIterator(
            docids=docids,
            freqs=freqs,
            doclens=self.doclens,
            N=self.N,
            avgdl=self.avgdl,
            k=self.K,
            b=self.B
        )

    def get_termids(self, tokens):
        return [self.lexicon[token][0] for token in tokens if token in self.lexicon]

    def get_postings(self, termids):
        return [self.get_posting(termid) for termid in termids]

#### **Purpose of Cell**:
This defines an `InvertedIndex` class to create, store, and access an index for the document retrieval system based on the **BM25 scoring model**.

#### **Theoretical Foundation**:
1. **Inverted Index**:
   - An **inverted index** maps terms to their corresponding document IDs. This data structure reduces search space significantly by directly pointing to documents where the given terms exist.
   - Example: If term `football` appears in documents `A`, `B`, and `C`, the inverted index for `football` would look like: `football → [A, B, C]`.

2. **BM25 Scoring Formula**:
   - BM25 is a widely-used ranking function in **information retrieval** to rank documents based on query relevance. It's derived from the **probabilistic retrieval model**.
   - Formula (simplified here):
     $$
     \text{BM25}(i) = \log \frac{N}{df_i} \cdot \frac{tf_i}{k \cdot ((1 - b) + b \cdot dl_i / avgdl) + tf_i}
     $$
     where:
     - $ tf_i $: Term frequency of $ i $ in the document.
     - $ df_i $: Document frequency of $ i $ (number of docs that contain $` i $).
     - $` dl_i `$: Document length.
     - $` avgdl `$: Average document length.
     - $` N `$: Total number of documents.
     - $` k, b `$: Hyperparameters controlling term saturation and length normalization.

3. **Posting List Iterator**:
   - Provides efficient sequential or target-based traversal of the inverted index. This is essential for implementing **document-at-a-time (DAAT)** query processing.

In [9]:
class TopQueue:
    def __init__(self, k=10, threshold=0.0):
        self.queue = []
        self.k = k
        self.threshold = threshold

    def size(self):
        return len(self.queue)

    def would_enter(self, score):
        return score > self.threshold

    def clear(self, new_threshold=None):
        self.queue = []
        if new_threshold:
            self.threshold = new_threshold

    def __repr__(self):
        return f'<{self.size()} items, th={self.threshold} {self.queue}'

    def insert(self, docid, score):
        if score > self.threshold:
            if self.size() >= self.k:
                heapq.heapreplace(self.queue, (score, docid))
            else:
                heapq.heappush(self.queue, (score, docid))
            if self.size() >= self.k:
                self.threshold = max(self.threshold, self.queue[0][0])
            return True
        return False

#### **Purpose of Cell**:
Defines a priority queue to maintain the **top-K search results** during query processing.

#### **Theoretical Foundation**:
1. **Heaps for Efficiency**:
   - Uses a **min-heap** data structure for efficient insertion and replacement of elements.
   - Maintains the property that the smallest score is always at the root, allowing quick comparison when determining if a new result qualifies to enter the top K list.

2. **Threshold Management**:
   - The `threshold` ensures that only documents with a score higher than the minimum of the top K results enter the heap.
   - This avoids unnecessary computation for low-scoring documents, improving scalability for queries over large indexes.

In [10]:
def min_docid(postings):
    min_docid = math.inf
    for p in postings:
        if not p.is_end_list():
            min_docid = min(p.docid(), min_docid)
    return min_docid

def daat(postings, k=10):
    top = TopQueue(k)
    current_docid = min_docid(postings)
    
    while current_docid != math.inf:
        score = 0
        next_docid = math.inf
        for posting in postings:
            if posting.docid() == current_docid:
                score += posting.score()
                posting.next()
            if not posting.is_end_list():
                next_docid = min(posting.docid(), next_docid)
        top.insert(current_docid, score)
        current_docid = next_docid
    return sorted(top.queue, reverse=True)

def query_process(query, index, heap_size=10):
    qtokens = set(preprocess(query))
    qtermids = index.get_termids(qtokens)
    postings = index.get_postings(qtermids)
    return daat(postings, heap_size)

#### **Purpose of Cell**:
Implements **document-at-a-time (DAAT)** query processing to score and rank documents based on BM25.

#### **Theoretical Foundation**:
1. **DAAT Algorithm**:
   - Processes documents **one at a time**, accessing all postings for query terms simultaneously.
   - Avoids scoring irrelevant documents, focusing only on those containing query terms.

2. **Efficient Ranking**:
   - The cross-comparison of successive `min_docid` ensures that all relevant documents are considered fairly and in sequence.

3. **Information Retrieval**:
   - Uses the inverted index and BM25 formulas to compute scores for query-document matching.

In [ ]:
def daat_with_weights(postings, k=10, weighted = False):
    top = TopQueue(k)
    posting_lists = postings.keys()
    current_docid = min_docid(posting_lists)
    
    while current_docid != math.inf:
        score = 0
        next_docid = math.inf
        for posting, weight in postings.items():
            if posting.docid() == current_docid:
                score += posting.score() * weight # moltiplico per il peso nel caso di nuovo vettore
                posting.next()
            if not posting.is_end_list():
                next_docid = min(posting.docid(), next_docid)
        top.insert(current_docid, score)
        current_docid = next_docid
    return sorted(top.queue, reverse=True)

def query_process_with_rocchio(query, index, heap_size=10, max_rel_docs=5):
    qtokens = set(preprocess(query))
    qtermids = index.get_termids(qtokens)
    postings = index.get_postings(qtermids)
    res = daat(postings, max_rel_docs)
    
    relevant_docids = [r[1] for r in res]
    
    new_query_terms = rocchio_pseudo_feedback(qtokens, relevant_docids, index.lexicon, index.direct)
    qtermids = index.get_termids(new_query_terms.keys())
    postings = index.get_postings(qtermids)
    
    weighted_postings = {}
    for term_weight, posting in zip(new_query_terms.values(), postings):
        weighted_postings[posting] = term_weight
    top_terms = dict(sorted(weighted_postings.items(), key=lambda item: item[1], reverse=True)[:10]) # solo i primi 10 termini
    
    return daat_with_weights(top_terms, heap_size)

#### **Purpose of Cell**:
Enhances the DAAT algorithm by:
1. **Incorporating weights** to adjust scores based on term relevance.
2. Using the **Rocchio Algorithm** to perform **pseudo-relevance feedback**, reformulating the query to improve retrieval performance.

#### **Theoretical Foundation**:
1. **Weights in Retrieval**:
   - Incorporates term weights to prioritize terms deemed more relevant by user interaction or a feedback mechanism (like Rocchio).
   - Adjusts scores dynamically by multiplying term-specific weights.

2. **Rocchio's Algorithm**:
   - A classical **relevance feedback method** that modifies the original query using relevant documents retrieved from the first round.
   - Formula:
     $$
     Q = \alpha \cdot Q_0 + \beta \cdot \left( \frac{1}{|D_R|} \sum_{d \in D_R} V(d) \right)
     $$
     - $ Q_0 $: Original query vector.
     - $ D_R $: Set of pseudo-relevant documents.
     - $ V(d) $: Vector of terms in document $ d $.
     - $ \alpha, \beta $: Hyperparameters adjusting weight contributions of query and document vectors.

3. **Pseudo-Relevance Feedback (PRF)**:
   - Assumes that the top K retrieved documents are relevant (since the system does not know relevance a priori).
   - Helps refine the query vector ($ Q $) to better match potential results.

In [12]:
def create_run_file(dataset, inv_index, name):
    topics = dataset.get_topics("test-2020")
    k = 1000
    results = []

    if inv_index:
        for query in tqdm(topics.itertuples(index=False), desc='Creating run file', total=len(topics)):
            res, query_id = query_process(query.query, inv_index, k), query.qid
            results.extend([f"{query_id} Q0 {r[1]} {rank} {r[0]} run.txt\n" for rank, r in enumerate(res, start=1)])
    else:
        for result in tqdm(topics.itertuples(index=False), desc='Creating run file', total=len(topics)):
            query_id, docno, rank, score = result.qid, result.docno, result.rank, result.score
            results.append(f"{query_id} Q0 {docno} {rank} {score} run.txt\n")

    with open(name, "w") as f:
        f.writelines(results)

def create_run_file_with_rocchio(dataset, inv_index, name, max_rel_docs=5):
    dataset = dataset.get_topics("test-2020")
    k = 1000
    results = []

    for query in tqdm(dataset.itertuples(index=False), desc='Creating run file', total=len(dataset)):
        res, query_id = query_process_with_rocchio(query.query, inv_index, k, max_rel_docs), query.qid
        results.extend([f"{query_id} Q0 {r[1]} {rank} {r[0]} run.txt\n" for rank, r in enumerate(res, start=1)])

    with open(name, "w") as f:
        f.writelines(results)

def create_qrels_file(dataset, name):
    qrels_df = dataset.get_qrels("test-2020")
    qrels = []

    for qrel in tqdm(qrels_df.itertuples(index=False), desc='Creating qrels file', total=len(qrels_df)):
        query_id, docno, label = qrel.qid, qrel.docno, qrel.label
        qrels.append(f"{query_id} 0 {docno} {label}\n")

    with open(name, "w") as f:
        f.writelines(qrels)

#### **Purpose of Cell**:
1. Creates `.run` files:
   - These represent the ranked retrieval results in TREC format for evaluation.
   - `.run` files store relevance rankings of document-query pairs.
2. Creates `.qrels` files:
   - These represent the ground truth relevance judgments for query-document pairs.

#### **Theoretical Foundation**:
1. **TREC Retrieval Evaluation**:
   - TREC format is a standard representation for ranking and relevance evaluation in Information Retrieval.
   - Format of `.run` file: 
     $$
     <query\_id> \; Q0 \; <docno> \; <rank> \; <score> \; <run\_tag>
     $$
   - Each run result contains:
     - `query_id`: The ID of the query.
     - `docno`: The ID of the document retrieved.
     - `rank`: The rank of the document produced by the system.
     - `score`: The relevance score assigned by the retrieval function.

2. **Evaluation Metrics**:
   **Qrels** files are critical for **ground truth** in calculating retrieval metrics like precision, recall, nDCG, etc., which compare system results (`.run` file) against known relevant documents.

3. **Pseudo Feedback in Query Refinements**:
   - The `create_run_file_with_rocchio` function uses Rocchio pseudo-relevance feedback to refine queries and generate more effective `.run` files.
   - Ground-truth `.qrels` are unaffected by refinements—they remain fixed as a point of comparison.

In [13]:
def trec_test(dataset=None, inv_index=None, text_file=None, rocchio=False, pseudo_rel_max_doc=0):
    if rocchio:
        create_run_file_with_rocchio((dataset, inv_index, pseudo_rel_max_doc, text_file))
    else:
        create_run_file(dataset, inv_index, text_file)

    qrels = list(ir_measures.read_trec_qrels('qrels.txt'))
    run = list(ir_measures.read_trec_run(text_file))
    
    print(ir_measures.calc_aggregate([P@5, P(rel=2)@5, nDCG@10, AP, AP(rel=2)], qrels, run))

    Pat5_list = {m.query_id: m.value for m in ir_measures.iter_calc([P@5], qrels, run)}
    PRel2at5_list = {m.query_id: m.value for m in ir_measures.iter_calc([P(rel=2)@5], qrels, run)}
    nDCGat10_list = {m.query_id: m.value for m in ir_measures.iter_calc([nDCG@10], qrels, run)}
    AP_list = {m.query_id: m.value for m in ir_measures.iter_calc([AP], qrels, run)}
    APRel2_list = {m.query_id: m.value for m in ir_measures.iter_calc([AP(rel=2)], qrels, run)}
    
    return Pat5_list, PRel2at5_list, nDCGat10_list, AP_list, APRel2_list

#### **Purpose of Cell**:
Calculates and outputs retrieval metrics (e.g., P@5 and nDCG@10) using `.run` and `.qrels` files.

#### **Theoretical Foundation**:
1. **Metrics in Information Retrieval**:
   - **Precision@N (P@N)**: Measures the proportion of relevant documents retrieved in the top-N results.
   - **nDCG@N (Normalized Discounted Cumulative Gain)**: Evaluates ranked results by prioritizing relevance at top positions.
   - **Average Precision (AP)**: Measures the area under the precision-recall curve, factoring in ranks.

2. **Pseudo-Relevance Feedback**:
   - Rocchio-adjusted queries (if enabled) aim to refine precision/recall by incorporating pseudo-relevance from the initial run.

In [14]:


def boxplot_and_results(data, metrics):
    for dicts, metric in zip(data, metrics):
        our_dict, baseline_dict = dicts[0], dicts[1]
        our_values = list(our_dict.values())
        baseline_values = list(baseline_dict.values())
    
        stat, p = stats.ttest_ind(our_values, baseline_values)
        print(f"T - test: statistic = {stat}, p = {p}")
    
        plt.figure(figsize=(8, 3))
        plt.title(metric)
        bplot = plt.boxplot([our_values, baseline_values],
                            vert=True,
                            patch_artist=True,
                            tick_labels=['Our', 'Baseline'])
        plt.show()

#### **Purpose of Cell**:
1. This function compares the performance metrics (e.g., Precision, nDCG) of two retrieval systems (e.g., "Our" implementation vs. "Baseline") using both statistical tests and visualizations.
2. It plots metrics using box plots and conducts statistical significance testing (via a t-test).

#### **Theoretical Foundation**:
1. **T-Test**:
   - A **two-sample t-test** is used to compare whether the mean performance of two retrieval systems is significantly different.
   - Null Hypothesis ($H_0$): The means of the performance metrics (e.g., P@5) for both datasets are statistically equal.
   - A low p-value (<0.05) indicates we can reject $H_0$, meaning there is a significant difference in retrieval performance.

2. **Box Plots**:
   - Box plots provide a summary of performance metrics, showing:
     - Median, interquartile range (IQR), and outliers.
   - Visualizes variability and central tendency of the metric values (e.g., how consistent the system performance is across queries).

3. **Metrics and Comparative Analysis**:
   - By inputting metrics like P@5, nDCG@10, or AP, it enables a deeper analysis of which system outperforms the other and in what capacity (e.g., better top rankings, cumulative relevance).

In [15]:
# TEST PER IL PREPROCESSING
queries = [
    "M4rk is a good programmer from U.S.A.%",
    "www.kaggle.com is a cool website, i love it%",
    "click this link for free 1000000 v-bucks, http://free-vbucks.com",
    "This is my new email, new_email@google.com",
    "Let`s go to this new Cafè, it's so good ♫ ☺%",
    "SPAM SPAM SPAM COPY COPY SPAM SPAM",
    "thissentenceisattachedbutwithwordninjawecansplitit"
]
for q in queries:
    print(preprocess(q))

['m4rk', 'good', 'programm', 'usa']
['cool', 'websit', 'love']
['click', 'link', 'free', '1000000', 'v', 'buck']
['new', 'email']
['let', 'go', 'new', 'cafe', 'good']
['spam', 'copi', 'spam']
['sentenc', 'attach', 'word', 'ninja', 'split']


#### **Purpose of Cell**:
Tests the `preprocess` function on a variety of sample textual inputs to evaluate how effectively it cleans and tokenizes the text.

#### **Theoretical Foundation**:
1. **Tokenization**:
   - The process of breaking down text into smaller units, such as words or phrases.
   - Important for search engine indexing since semantic similarity is determined at the token level.

2. **Text Cleaning**:
   - Removes irrelevant content such as URLs, email addresses, and special characters to standardize text for information retrieval.
   - Example:
     - Input: `"This is my email, new_email@google.com"`
     - Output: `["email", "new", "email", "googl"]` (stemmed and cleaned).

3. **Word Splitting (Long Tokens)**:
   - Uses `wordninja` to split concatenated tokens when they exceed 10 characters. This is useful for compound words or typos (e.g., `"thissentenceisattached"` gets split into `["this", "sentence", "is", "attached"]`).

4. **Stopword Removal**:
   - Eliminates frequently occurring but less meaningful words (e.g., "is", "the") to focus on essential semantic content.

In [16]:

# TEST TO SEE IF THE BM25 FORMULA IMPLEMENTED IS CORRECT
test_docs = [['The cat playing on the bed', '1'], ['Nick is playing football', '2'], ['Mark is whatching football on the bed', '3']]
test_df = pd.DataFrame(test_docs, columns=['text', 'docno'])
print(test_df)
print()

test_df["tokens"] = test_df["text"].apply(preprocess)
print(test_df)

t_lex, t_inv, t_doc, t_direct, t_stats = build_index(test_df)
print(t_lex)
print(t_inv)
print(t_doc)
print(t_direct)
print(t_stats)

                                    text docno
0             The cat playing on the bed     1
1               Nick is playing football     2
2  Mark is whatching football on the bed     3

                                    text docno                        tokens
0             The cat playing on the bed     1              [cat, play, bed]
1               Nick is playing football     2         [nick, play, footbal]
2  Mark is whatching football on the bed     3  [mark, whatch, footbal, bed]


Indexing: 100% 3/3 [00:00<00:00, 12312.05it/s]

{'cat': [0, 1, 1], 'play': [1, 2, 2], 'bed': [2, 2, 2], 'nick': [3, 1, 1], 'footbal': [4, 2, 2], 'mark': [5, 1, 1], 'whatch': [6, 1, 1]}
{'docids': {0: [1], 1: [1, 2], 2: [1, 3], 3: [2], 4: [2, 3], 5: [3], 6: [3]}, 'freqs': {0: [1], 1: [1, 1], 2: [1, 1], 3: [1], 4: [1, 1], 5: [1], 6: [1]}}
[('1', 3), ('2', 3), ('3', 4)]
{1: {0: 1, 1: 1, 2: 1}, 2: {3: 1, 1: 1, 4: 1}, 3: {5: 1, 6: 1, 4: 1, 2: 1}}
{'num_docs': 3, 'num_terms': 7, 'num_tokens': 10}


#### **Purpose of Cell**:
Validates the correctness of preprocessing, tokenization, and inverted index construction.
- Works with a small, test dataset (3 documents) to ensure functionality and debug any issues.

#### **Theoretical Foundation**:
1. **Index Structures**:
   - **Lexicon**:
     - Maps terms to unique IDs.
     - Tracks term frequencies across the collection and the number of documents containing the term.
   - Example: `football -> [1, 2, 3]`, where:
     - `1`: Term ID,
     - `2`: Document frequency (DF),
     - `3`: Term Frequency (TF).

   - **Inverted Index**:
     - Efficiently maps terms to the documents they appear in. 
     - Example: `football -> { [2, 3] }` (IDs of documents containing "football").

   - **Statistics**:
     - Total tokens, unique terms, and document counts help validate that indexing works correctly.

2. **Testing BM25 Calculations**:
   - BM25 relies on intermediate data from the generated index (term frequencies, document lengths, etc.).
   - This cell ensures preprocessing and indexing set up the required conditions for accurate document retrievability.

In [17]:
query = "Nick football"
print(preprocess(query))
# total doc length = 10
# avg doc length = 3.333
# k = 1.2 as eg.
# b = 0.7 as eg.
# math.log è in base e
# doc 1 -> BM25 = 0
# doc 2 -> BM25 = [ Nick, (1 * log(3/1))/(1.2*((1-0.7) + 0.7*(3/3.33)) + 1) = 0.519; 
#                  football, (1 * log(3/2))/(1.2*((1-0.7) + 0.7*(3/3.33)) + 1) = 0.1915] = 0.7105
# doc 3 -> BM25 = [ football, (1 * log(3/2))/(1.2*((1-0.7) + 0.7*(4/3.33)) + 1) ] = 0.171
test_inv_index = InvertedIndex(t_lex, t_inv, t_doc, t_direct, t_stats, 1.2, 0.7)
res = query_process(query, test_inv_index, 10)
res # i risultati combaciano

['nick', 'footbal']


[(0.7108116241853848, 2), (0.1712268193024343, 3)]

#### **Purpose of Cell**:
Manually assesses the correctness of the **BM25 scoring** computation by working through a small dataset and query.

#### **Theoretical Foundation**:
1. **BM25 Calculations**:
   - Demonstrates how term frequency, document frequency, document lengths, and parameters \( k, b \) are combined to generate a ranking score.
   - Calculations are explicitly written out for detailed step-by-step validation.

2. **Relevance**:
   - Shows how different documents contribute to query relevance scores, depending on term matches and length normalization.

3. **Testing Framework**:
   - Verifies not only scores but also that preprocessing, tokenizing, and term-to-document mappings propagate correctly through the system.


In [18]:

tsv_file = 'collection.tsv'

# Load large-scale collection data using Polars and convert to DataFrame
temp = pl.read_csv(tsv_file, separator='\t', has_header=False, new_columns=['docno', 'text'], encoding="utf-8").to_pandas()

#### **Purpose of Cell**:
Loads the MS MARCO `collection.tsv` file containing millions of passages into memory for indexing.

#### **Theoretical Foundation**:
1. **Efficient File Reading**:
   - MS MARCO is a large-scale dataset, and using `Polars` provides an optimized way of handling and processing large `.tsv` files.
   - Polars is known for its speed compared to traditional libraries like `pandas`.

2. **TSV Structure**:
   - The dataset is a tab-separated file where:
     - `docno`: A unique document/passage ID.
     - `text`: Raw passage content.
   - The goal is to preprocess and index the `text` field for retrieval.

3. **Data Conversion**:
   - Once read by Polars, the dataset is converted to a pandas DataFrame for compatibility with the rest of the indexing and retrieval system.

#### **Key Note**:
- This operation may require sufficient memory, as reading and processing the MS MARCO dataset can consume several gigabytes of RAM. On limited systems, consider chunk-based reading.

In [19]:
temp["tokens"] = temp["text"].progress_apply(preprocess)

stem_word.cache_clear()
split_word.cache_clear()

Preprocessing...: 100% 8841823/8841823 [09:11<00:00, 16024.49it/s] 


#### **Purpose of Cell**:
1. Applies the `preprocess` function to every text field in the dataset.
2. Caches from the `stem_word` and `split_word` functions are explicitly cleared to prevent stale entries when processing large datasets.

#### **Theoretical Foundation**:
1. **Batch Preprocessing**:
   - Applies the same preprocessing workflow previously tested (in Cell 14). This is done for all passages in the collection to ensure uniform tokenization and standardization.
   - Key steps include normalization, stemming, stopword removal, and tokenization.

2. **Caching Efficiency**:
   - The `stem_word` and `split_word` functions are cache-heavy due to the use of `@lru_cache`. This ensures frequently queried tokens reuse their precomputed values.
   - Clearing the cache improves memory usage after processing a large dataset.

3. **Scalability via `tqdm`**:
   - The `.progress_apply()` function gives real-time feedback on progress. This is especially helpful when preprocessing millions of records, providing a clear estimation of completion time.

In [20]:
lex, inv, doc, direct, stats = build_index(temp)
print(stats)
# save data in a file to avoid reindexing

with open('data.pkl', 'wb') as f:
    pickle.dump((lex, inv, doc, direct, stats), f)



Indexing: 100% 8841823/8841823 [05:54<00:00, 24929.52it/s]


{'num_docs': 8841823, 'num_terms': 1205151, 'num_tokens': 307062371}


#### **Purpose of Cell**:
1. Constructs the full inverted index from the preprocessed dataset for retrieval.
2. Saves the generated indices to a file (`data.pkl`) for future use, avoiding the need to rebuild.

#### **Theoretical Foundation**:
1. **Index Construction**:
   - Uses the `build_index` function (defined in Cell 6) to compute:
     - `lexicon`: Maps terms to term IDs and term/document frequencies.
     - `inverted index`: Tracks documents containing terms and their frequencies.
     - `direct index`: Tracks term frequencies for individual documents.
     - `doc index`: Tracks document lengths.
     - `stats`: Aggregates dataset-level information (e.g., total terms, number of tokens, etc.).

2. **Serialization and Persistence**:
   - **Pickle**:
     - Python's built-in library for saving objects to a binary file (`data.pkl` in this case).
     - Enables quick reloading of the index without repeating preprocessing and indexing steps during subsequent runs.
   - Benchmark:
     - This saves hours of preprocessing and memory-intensive tasks, making the workflow efficient for iterative testing.

3. **Large-Scale Impact**:
   - Handles datasets with millions of rows for real-world applications of large-scale retrieval tasks.

In [22]:
# check if lexicon and other files are present
try:
    with open('data.pkl', 'rb') as f:
        lex, inv, doc, direct, stats = pickle.load(f)
except FileNotFoundError:
    print("File not found")
    

inv_lexicon = {v[0]: k for k, v in lex.items()} # Per ottenere come risultato una query di stringhe e non numeri

with open(os.path.join("files", "lex.pkl"), "wb") as f:
    joblib.dump(lex, f)
    del lex
gc.collect()

# Parametri di Rocchio
alpha = 8
beta = 16

def rocchio_pseudo_feedback(query_terms, doc_ids, lexicon, direct_index):
    # il seguente codice svolge l'algoritmo di rocchio senza tenere in considerazione i termini dei
    # doc non rilevanti (gamma = 0). inoltre la restrizione viene fatta successivamente sui primi 10
    # termini con peso maggiore. Il parametro che stiamo variando nell'ottimizzazione è il numero di documenti
    # tra quelli restituiti dal primo search da considerare rilevanti, dato che è una pseudo relevance feedback
    
    query_vector = defaultdict(int) # creazione del vettore della query
    for term in set(query_terms): # utilizzo set in quanto ottimizzo la ricerca, tanto userò peso unitario per iniziare
        if term in lexicon: # i termini sconosciuti assegnamo peso 0
            query_vector[lexicon[term][0]] = 1 # idicizziamo tramite term id

    relevant_vector = defaultdict(float) # creazione del vettore dei doc rilevanti
    doc_set = set(doc_ids) # per motivi di ottimizzazione passiamo ad un set, come fatto per i termini della query
            
    for doc in doc_set:
        for term_id, tf in direct_index[doc].items():
            relevant_vector[term_id] += tf
    # ciò che fa il ciclo for al di sopra consiste nel compattare la sommatoria sui documenti rilevanti in un dizionario
    # dove per ogni termine sommiamo le frequenze in tutti i documenti nel doc_set, ovvero i
    # documenti ritenuti rilevanti, in modo da poterlo direttamente sommare al query_vector
    
    new_query_vector = defaultdict(float)
    for term_id in query_vector.keys() | relevant_vector.keys(): # iteriamo su tutti i nuovi termini
        new_query_vector[inv_lexicon[term_id]] = (alpha * query_vector[term_id]) + (beta * relevant_vector[term_id] / len(doc_ids))

    return new_query_vector

#### **Purpose of Cell**:
1. Tries to load precomputed index files to save computational time.
2. If no file is found, alerts the user to rebuild the index from scratch.
3. **Optimizes memory** by serializing individual components of the index using `joblib`.
4. Implements **Rocchio's Algorithm for Pseudo-Relevance Feedback**, which reformulates queries to improve retrieval results.

#### **Theoretical Foundation**:
1. **Prebuilt Index Advantages**:
   - Once `data.pkl` exists, the system avoids unnecessary recomputation of the index. This saves hours of preprocessing and indexing for large datasets.

2. **Garbage Collection (GC)**:
   - Large datasets occupy considerable memory, often leading to out-of-memory issues. Triggering garbage collection frees unnecessary memory and improves performance for subsequent operations.

3. **Interpretable Lexicon**:
   - The reverse mapping (`inv_lexicon`) enables converting term IDs back into actual terms. This is crucial for debugging, displaying query results, or understanding term statistics.

4. **What is `Joblib`?**:
   - Joblib is optimized for serializing large objects in Python, making it more memory-efficient than `pickle` for complex data structures.

5. **Rocchio's Algorithm** (see Cell 10):
   - Adjusts the original query representation by combining it with the aggregated term usage of documents assumed to be relevant.
   - Hyperparameters:
     - \( \alpha \): Weight for the original query terms.
     - \( \beta \): Weight for the relevant document contributions.

6. **Query Refinement**:
   - Results in a weighted query vector including both original and new terms.
   - **Example**: If "football" is highly frequent across pseudo-relevant documents, it receives higher weight and is prioritized for subsequent searches.

7. **Pseudo-Relevance Feedback**:
   - Assumes the top \( K \) retrieved documents are relevant, which may not always be true. Careful \( K \)-selection reduces noise introduced in the reformulated query.

In [ ]:

# Ensure the 'files' directory exists
os.makedirs("files", exist_ok=True)

with open(os.path.join("files", "inv.pkl"), "wb") as f:
    joblib.dump(inv, f)
    del inv
with open(os.path.join("files", "doc.pkl"), "wb") as f:
    joblib.dump(doc, f)
    del doc
with open(os.path.join("files", "direct.pkl"), "wb") as f:
    joblib.dump(direct, f)
    del direct
with open(os.path.join("files", "stats.pkl"), "wb") as f:
    joblib.dump(stats, f)
    del stats
gc.collect()

#### **Purpose of Cell**:
1. After loading or rebuilding parts of the dataset (from `data.pkl`):
   - Separately saves the components (`inv`, `doc`, `direct`, `stats`) using `joblib` for modular storage.
   - Frees memory by deleting individual variables right after serialization.
2. Enhances modularity for future use by creating easily reloadable files for specific components of the retrieval system.

---

#### **Theoretical Foundation**:
1. **Distributed Storage for Scalability**:
   - By splitting and saving index components individually:
     - Easier to load only the parts needed (e.g., inverted index alone for querying).
     - Reduces memory overhead during retrieval tasks.

2. **Garbage Collection**:
   - Manages memory more effectively on systems with limited resources or large datasets.
   - For example:
     - MS MARCO passages can consume memory in gigabytes; clearing unused data is crucial to avoid memory exhaustion.

In [ ]:
with open(os.path.join("files", "lex.pkl"), "rb") as f:
    lex = joblib.load(f)
with open(os.path.join("files", "inv.pkl"), "rb") as f:
    inv = joblib.load(f)
with open(os.path.join("files", "doc.pkl"), "rb") as f:
    doc = joblib.load(f)
with open(os.path.join("files", "direct.pkl"), "rb") as f:
    direct = joblib.load(f)
with open(os.path.join("files", "stats.pkl"), "rb") as f:
    stats = joblib.load(f)
    doc = joblib.load(f)
with open(os.path.join("files", "direct.pkl"), "rb") as f:
    direct = joblib.load(f)
with open(os.path.join("files", "stats.pkl"), "rb") as f:
    stats = joblib.load(f)

#### **Purpose of Cell**:
Quickly reloads the previously saved components of the index from their modular serialized files.

#### **Theoretical Foundation**:
1. **Component-Based Reloading**:
   - By saving and loading files separately (`lex.pkl`, `inv.pkl`, etc.), the system gains flexibility.
   - Examples:
     - Load only `lex.pkl` to inspect token mappings.
     - Load `stats.pkl` to check dataset statistics for analytical tasks.

2. **Serialization Benefits**:
   - Uses `save-load` cycles to avoid rebuilding the index for every query or system restart.
   - Drastically reduces time-to-load, as rebuilding the index can take several hours for large datasets like MS MARCO.

In [ ]:
inv_index = InvertedIndex(lex, inv, doc, direct, stats, 1.2, 0.7)

with open(os.path.join("files", "inv_index.pkl"), "wb") as f:
    joblib.dump(inv_index, f)

del lex, inv, doc, direct, stats
gc.collect()

#### **Purpose of Cell**:
1. Combines the saved index components (`lex`, `inv`, `doc`, `direct`, `stats`) into a single instance of the `InvertedIndex` class.
2. Serializes the complete `InvertedIndex` object (`inv_index`) as `inv_index.pkl` for streamlined querying and retrieval.

---

#### **Theoretical Foundation**:

1. **Parametric Configuration of BM25**:
   - This cell sets up the BM25 parameters (`k=1.2`, `b=0.7`) for ranking. These influence:
     - Term saturation: \( k \) controls the term frequency scaling.
     - Length normalization: \( b \) adjusts for document length discrepancies relative to the average length.

2. **Reusable Data Structures**:
   - The serialized `inv_index` file:
     - Encapsulates all required components for querying.
     - Makes it easy to reload and run end-to-end queries without accessing individual subcomponents (`lex`, `inv`, etc. separately).

In [ ]:


dataset = pt.datasets.get_dataset("msmarco_passage") # per le query
create_qrels_file(dataset, "qrels.txt")

#### **Purpose of Cell**:
1. Loads the complete MS MARCO passage dataset (using PyTerrier's `datasets` module).
2. Creates the `.qrels` file for evaluation by extracting ground-truth relevance data.

---

#### **Theoretical Foundation**:

1. **The MS MARCO Benchmark**:
   - MS MARCO (Microsoft MAchine Reading COmprehension) is a standard dataset for evaluating passage retrieval systems.
   - Contains:
     - A vast collection of passages (`collection.tsv`).
     - Queries and their top-K relevance judgments (`qrels.tsv`).

2. **Ground Truth (`qrels.txt`)**:
   - `.qrels` files link queries to relevant documents using known human-labeled relevance scores.
   - Example format:
     \[
     query\_id \; 0 \; document\_id \; relevance\_label
     \]
     - `relevance_label`: Values like `0` (non-relevant) or higher (`1`, `2`, etc.) indicating relevance.

3. **Dataset Availability via PyTerrier**:
   - PyTerrier automates dataset fetching and caching, minimizing the need for manual file downloads.

In [ ]:
# PYTERRIER TEST #
pt.init()

bm25_terrier_stemmed = pt.terrier.Retriever.from_dataset('msmarco_passage', 'terrier_stemmed', wmodel='BM25')

pipeline = bm25_terrier_stemmed
pipeline(dataset.get_topics("test-2020"))

#### **Purpose of Cell**:
Conducts a retrieval test using the **PyTerrier BM25 pipeline** as a baseline system.

---

#### **Theoretical Foundation**:
1. **PyTerrier Toolkit**:
   - PyTerrier is an information retrieval library that seamlessly integrates with the Terrier indexing and retrieval platform.
   - It provides standardized ways to:
     - Load datasets.
     - Conduct experiments using prebuilt retrieval pipelines like BM25.

2. **BM25 Configuration**:
   - Performs BM25-based retrieval with **stemming enabled**.
   - This contrasts against our internally implemented retrieval system (non-Terrier), providing a **benchmark for comparison**.

3. **Test Queries**:
   - Uses the `test-2020` topic queries from MS MARCO for evaluation.
   - `test-2020` topics consist of user queries refined by annotators into various topics/tasks.

In [ ]:
OurPat5, OurPRel2at5, OurnDCGat10, OurAP, OurAPRel2 = trec_test(dataset, inv_index, "run.txt")
BaselinePat5, BaselinePRel2at5, BaselinenDCGat10, BaselineAP, BaselineAPRel2 = trec_test(dataset, None, "ptrun.txt")

#### **Purpose of Cell**:
Adds a **side-by-side performance test** between:
- **Our Custom BM25 Retrieval System (`Our`)**.
- **Baseline PyTerrier Query Processing Pipeline (`Baseline`)**.

---

#### **Theoretical Foundation**:
1. **Metrics Computation**:
   - Invokes the prewritten `trec_test()` function (Cell 12) to compute retrieval metrics:
     - P@N: Precision at cutoff \( N \).
     - nDCG@10: Quality of the ranking order, giving higher weight to top results.
     - AP: Average Precision, representing area under the precision-recall curve.

2. **Evaluation File Usage**:
   - The `run.txt` file is generated from our implementation.
   - The `ptrun.txt` file represents results produced by PyTerrier's `terr_stemmed` pipeline.

3. **Direct Comparison**:
   - Captures output from both systems into results (`Our...` vs. `Baseline...`) for future statistical analysis and plotting.


In [ ]:
data = [(OurPat5, BaselinePat5),
        (OurPRel2at5, BaselinePRel2at5),
        (OurnDCGat10, BaselinenDCGat10),
        (OurAP, BaselineAP),
        (OurAPRel2, BaselineAPRel2)]
metrics = ["P@5", "P(rel=2)@5", "nDCG@10", "AP", "AP(rel=2)"]

boxplot_and_results(data, metrics)

#### **Purpose of Cell**:
1. Analyzes retrieval system results by:
   - Comparing our implementation (`Our...` metrics) against the baseline PyTerrier retrieval system (`Baseline... metrics`).
   - Using box plots and statistical evaluations (via a t-test) to assess the differences.
2. Evaluates five major metrics: P@5, P(rel=2)@5, nDCG@10, AP, and AP(rel=2).

---

#### **Theoretical Foundation**:
1. **Visualizing Performance**:
   - Box plots provide a view of the:
     - **Distribution of scores** for the respective metrics across queries.
     - Identify outliers, medians, and variability within and between systems.
   - Focuses on ranking precision (via nDCG), ranking order (AP), and early precision (P@5 and P(rel=2)@5).

2. **Significance Tests (Two-Sample T-Test)**:
   - The paired t-test determines if observed differences in metric distributions are statistically significant.
   - Null Hypothesis (\(H_0\)): There is no difference in performance between the two retrieval systems.
   - A low p-value (e.g., p < 0.05) indicates the null hypothesis can be rejected, suggesting meaningful performance differences.

3. **Comparative Outcome**:
   - Helps assess whether our system outperforms, matches, or underperforms relative to an established PyTerrier baseline.
   - Directly informs iterative system improvements (e.g., modifying BM25 \(k, b\) parameters or refining query processing).

In [ ]:
RocchioPat5, RocchioPRel2at5, RocchionDCGat10, RocchioAP, RocchioAPRel2 = trec_test(dataset, inv_index, "run.txt", True, 3)

new_data = [(RocchioPat5, BaselinePat5),
            (RocchioPRel2at5, BaselinePRel2at5),
            (RocchionDCGat10, BaselinenDCGat10),
            (RocchioOurAP, BaselineAP),
            (RocchioOurAPRel2, BaselineAPRel2)]

boxplot_and_results(new_data, metrics)

#### **Purpose of Cell**:
1. Introduces **Rocchio feedback scoring** into the retrieval process.
2. Compares metrics with and without pseudo-relevance feedback alongside the baseline.
3. Generates updated performance visualizations and significance tests.

---

#### **Theoretical Foundation**:
1. **Effect of Rocchio Feedback**:
   - Rocchio pseudo-relevance feedback is expected to:
     - Improve Recall: By expanding the query to include terms from pseudo-relevant documents.
     - Potentially Compromise Precision: By introducing noisy terms if pseudo-relevant documents contain irrelevant content.

2. **Metric Behavior with Rocchio**:
   - Metrics like nDCG and AP are sensitive to ranking order, meaning Rocchio can significantly impact them.
   - P@5 (precision among top results) assesses if Rocchio refinements push relevant content early in the ranking.

3. **New Testing Scope**:
   - The addition of Rocchio expands the comparative landscape, allowing analysis of the retrieval system **with feedback**, **without feedback**, and **against the baseline PyTerrier system**.


In [ ]:
def profile(f):
    def f_timer(*args, **kwargs):
        start = time.time()
        result = f(*args, **kwargs)
        end = time.time()
        ms = (end - start) * 1000
        print(f"{f.__name__} ({ms:.3f} ms)")
        return result
    return f_timer

#### **Purpose of Cell**:
Defines a decorator function to **measure and profile execution time** for other functions in milliseconds.

---

#### **Theoretical Foundation**:
1. **Performance Profiling**:
   - Execution speed is critical for real-world efficiency in **retrieval systems**, particularly when querying large datasets like MS MARCO.
   - Helps identify bottlenecks in:
     - Query preprocessing.
     - Posting list traversal.
     - BM25 scoring.
   - Results can guide future optimizations or parallelization efforts for scalability.

2. **Decorator Pattern**:
   - Decorators in Python enable modifying the behavior of a function (e.g., adding execution timing) without altering its source code.
   - The `profile` decorator wraps a function, measures its time, and prints the results.


In [ ]:
@profile
def timed_query_process(query, index, heap_size=10):
    qtokens = set(preprocess(query))
    qtermids = index.get_termids(qtokens)
    postings = index.get_postings(qtermids)
    return daat(postings, heap_size)

@profile
def timed_query_process_with_rocchio(query, index, heap_size=10, max_rel_docs=5):
    qtokens = set(preprocess(query))
    qtermids = index.get_termids(qtokens)
    postings = index.get_postings(qtermids)
    res = daat(postings, max_rel_docs)
    
    relevant_docids = [r[1] for r in res]
    
    new_query_terms = rocchio_pseudo_feedback(qtokens, relevant_docids, index.lexicon, index.direct)
    qtermids = index.get_termids(new_query_terms.keys())
    postings = index.get_postings(qtermids)
    
    weighted_postings = {}
    for term_weight, posting in zip(new_query_terms.values(), postings):
        weighted_postings[posting] = term_weight
    top_terms = dict(sorted(weighted_postings.items(), key=lambda item: item[1], reverse=True)[:10])
    
    return daat_with_weights(top_terms, heap_size)

#### **Purpose of Cell**:
Profiles the execution time for:
1. **Standard Query Processing** (without Rocchio).
2. **Query Processing with Rocchio Feedback**.

---

#### **Theoretical Foundation**:
1. **Timing Retrieval Operations**:
   - Non-Rocchio queries are expected to be faster due to fewer adjustments (no query reformulation or relevance feedback processing).
   - Rocchio processing introduces a second retrieval step and term weight computations, increasing execution time.

2. **Efficiency Evaluation**:
   - Execution profiles provide direct insight into trade-offs between effectiveness (improved metrics) and efficiency (processing time).

3. **Scalability**:
   - Enables testing how both modes scale with query complexity, data volume, or the number of feedback terms.

In [ ]:
query_results = dict()

for query in dataset.get_topics("test-2020")[:5].itertuples(index=False):
    res = timed_query_process(query.query, inv_index)
    query_results[query.query] = []
    for r in res:
        row = temp.loc[temp["docno"] == r[1]]
        query_results[query.query].append((row.docno.values, row.text.values))

for q, res in query_results.items():
    print(q)
    print()
    for r in res:
        print(r[1])
    print("----------")

#### **Purpose of Cell**:
1. Executes the non-Rocchio query retrieval for the first 5 topics in `test-2020`.
2. Displays the retrieved documents and their text content for manual inspection.

---

#### **Theoretical Foundation**:
1. **Result Validation**:
   - Allows manually verifying BM25 rankings against human intuition and expected document relevance.

2. **Practical Debugging**:
   - Helps trace potential errors or misbehavior in preprocessing (e.g., stemming errors) or scoring (e.g., out-of-place ranks).

In [ ]:
query_results = dict()

for query in dataset.get_topics("test-2020")[:5].itertuples(index=False):
    res = timed_query_process_with_rocchio(query.query, inv_index, max_rel_docs=3)
    query_results[query.query] = []
    for r in res:
        row = temp.loc[temp["docno"] == r[1]]
        query_results[query.query].append((row.docno.values, row.text.values))

for q, res in query_results.items():
    print(q)
    print()
    for r in res:
        print(r[1])
    print("----------")

#### **Purpose of Cell**:
Tests query retrieval using Rocchio-based pseudo-relevance feedback for the first 5 topics in the test set.

---

#### **Theoretical Foundation**:
1. **Feedback-Enhanced Relevance**:
   - Incorporates feedback from top pseudo-relevant documents into the query, enhancing term-coverage and retrieval results.
   - Demonstrates the immediate impact of Rocchio refinement on ranked outputs.

2. **Qualitative Inspection**:
   - Enables comparison of the difference in documents retrieved with vs. without feedback, showing if Rocchio made meaningful improvements.